# ਪਾਠ 11 - ਮਾਡਲ ਸੰਦਰਭ ਪ੍ਰੋਟੋਕੋਲ (MCP)

**ਮਾਡਲ ਸੰਦਰਭ ਪ੍ਰੋਟੋਕੋਲ (MCP)** ਇੱਕ ਖੁੱਲਾ ਮਿਆਰ ਹੈ ਜੋ ਏਜੰਟਾਂ ਨੂੰ ਰਨਟਾਈਮ ਵਿੱਚ ਗਤੀਸ਼ੀਲ ਤੌਰ 'ਤੇ ਟੂਲਾਂ, ਸਰੋਤਾਂ, ਅਤੇ ਡਾਟਾ ਸੋਰਸਾਂ ਨੂੰ ਖੋਜਣ ਅਤੇ ਵਰਤੋਂ ਦੀ ਆਗਿਆ ਦਿੰਦਾ ਹੈ। ਐਜੰਟ ਵਿੱਚ ਟੂਲਾਂ ਨੂੰ ਹੋਰਡਕੋਡ ਕਰਨ ਦੀ ਬਜਾਏ, MCP ਏਜੰਟਾਂ ਨੂੰ ਬਾਹਰੀ ਸਰਵਰਾਂ ਨਾਲ ਜੁੜਨ ਦਿੰਦਾ ਹੈ ਜੋ ਮੰਗ 'ਤੇ ਖੁਬੀਆਂ ਪ੍ਰਗਟਾਉਂਦੇ ਹਨ।

ਇਸ ਪਾਠ ਵਿੱਚ, ਤੁਸੀਂ ਸਿੱਖੋਗੇ:
- MCP ਕੀ ਹੈ ਅਤੇ ਇਹ ਏਜੰਟ ਸਿਸਟਮਾਂ ਲਈ ਕਿਉਂ ਮਹੱਤਵਪੂਰਨ ਹੈ
- MCP ਕਲਾਇੰਟ-ਸਰਵਰ ਆਰਕੀਟੈਕਚਰ ਕਿਵੇਂ ਕੰਮ ਕਰਦਾ ਹੈ
- MCP-ਸ਼ੈਲੀ ਟੂਲ ਖੋਜ ਦੀ ਵਰਤੋਂ ਕਰਨ ਵਾਲੇ ਏਜੰਟ ਬਣਾਉਣ ਦਾ ਤਰੀਕਾ


## ਸੈੱਟਅੱਪ

**ਜਰੂਰੀਆਂ:**
- ਮਾਇਕਰੋਸੌਫਟ ਫਾਊਂਡਰੀ ਪ੍ਰੋਜੈਕਟ ਜਿਸ ਵਿੱਚ ਡਿਪਲੌਇਡ ਮਾਡਲ ਹੋਵੇ
- ਪ੍ਰਮਾਣੀਕਰਨ ਲਈ `az login` ਚਲਾਓ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ਮਾਡਲ ਕਾਂਟੈਕਸਟ ਪ੍ਰੋਟੋਕੋਲ (MCP) ਕੀ ਹੈ?

MCP ਇੱਕ ਮਿਆਰੀ ਤਰੀਕਾ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦਾ ਹੈ ਜਿਸ ਨਾਲ AI ਏਜੰਟ ਬਾਹਰੀ ਟੂਲਾਂ ਅਤੇ ਡਾਟਾ ਸਰੋਤਾਂ ਨੂੰ ਖੋਜ ਅਤੇ ਇੰਟਰਐਕਟ ਕਰਦੇ ਹਨ:

- **MCP ਸਰਵਰ**: ਇੱਕ ਮਿਆਰੀ ਪ੍ਰੋਟੋਕੋਲ ਰਾਹੀਂ ਟੂਲ, ਸਰੋਤ ਅਤੇ ਪ੍ਰਾਂਪਟ ਦਿਖਾਉਂਦਾ ਹੈ
- **MCP ਕਲਾਇੰਟ**: ਉਹ ਏਜੰਟ ਰਨਟਾਈਮ ਜੋ ਸਰਵਰਾਂ ਨਾਲ ਜੁੜਦਾ ਹੈ ਅਤੇ ਉਪਲਬਧ ਸਮਰੱਥਾਵਾਂ ਨੂੰ ਖੋਜਦਾ ਹੈ
- **ਡਾਇਨਾਮਿਕ ਡਿਸਕਵਰੀ**: ਏਜੰਟਾਂ ਨੂੰ ਹਾਰਡਕੋਡਡ ਟੂਲਾਂ ਦੀ ਲੋੜ ਨਹੀਂ — ਉਹ ਰਨਟਾਈਮ 'ਤੇ ਜੋ ਉਪਲਬਧ ਹੁੰਦਾ ਹੈ ਉਹ ਖੋਜਦੇ ਹਨ

ਇਹ ਵਧੇਰੇ ਸਮਰੱਥਾ ਵਾਲੇ ਏਜੰਟ ਸਿਸਟਮ ਬਣਾਉਣ ਲਈ ਸ਼ਕਤਸ਼ਾਲੀ ਹੈ ਜਿੱਥੇ ਨਵੀਆਂ ਸਮਰੱਥਾਵਾਂ ਬਿਨਾਂ ਏਜੰਟ ਕੋਡ ਨੂੰ ਬਦਲੇ ਜੋੜੀਆਂ ਜਾ ਸਕਦੀਆਂ ਹਨ।


## MCP ਕਿਵੇਂ ਕੰਮ ਕਰਦਾ ਹੈ

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. ਏਜੰਟ (MCP ਕਲਾਇੰਟ) ਇੱਕ MCP ਸਰਵਰ ਨਾਲ ਜੁੜਦਾ ਹੈ
2. ਸਰਵਰ ਉਪਲਬਧ ਟੂਲਾਂ ਅਤੇ ਉਹਨਾਂ ਦੇ ਸਕੀਮਾਂ ਦੀ ਸੂਚੀ ਨਾਲ ਜਵਾਬ ਦਿੰਦਾ ਹੈ
3. ਏਜੰਟ ਫਿਰ ਆਪਣੇ ਤਰਕ ਵਿੱਚ ਕਿਸੇ ਵੀ ਮਿਲੇ ਹੋਏ ਟੂਲ ਨੂੰ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ
4. ਨਤੀਜੇ ਉਸੇ ਪ੍ਰੋਟੋਕੋਲ ਰਾਹੀਂ ਵਾਪਸ ਆਉਂਦੇ ਹਨ


## MCP ਟੂਲ ਦੀ ਖੋਜ ਦਾ ਅਨੁਕਰਨ

ਚੁੱਕੀ ਇੱਕ ਅਸਲ MCP ਸਰਵਰ ਨੂੰ ਚੱਲ ਰਹੀ ਸਰਵਰ ਪ੍ਰਕਿਰਿਆ ਦੀ ਲੋੜ ਹੁੰਦੀ ਹੈ, ਅਸੀਂ ਇਸ ਪੈਟਰਨ ਨੂੰ `@tool` ਫੰਕਸ਼ਨਾਂ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਦਰਸਾਵਾਂਗੇ ਜੋ MCP-ਸੰযুক্ত ਰਹਿਣ-ਸਹਾਇਤਾ ਸੇਵਾ ਕੀ પ્રદਾਨ ਕਰੇਗੀ ਉਸ ਦਾ ਅਨੁਕਰਨ ਕਰਦੇ ਹਨ।

ਉਤਪਾਦਨ ਵਿੱਚ, ਇਹ ਟੂਲ ਇੱਕ MCP ਸਰਵਰ ਤੋਂ ਗਤੀਸ਼ੀਲ ਤਰੀਕੇ ਨਾਲ ਖੋਜੇ ਜਾਣਗੇ, ਨਾ ਕਿ ਸਥਾਨਕ ਤੌਰ 'ਤੇ ਪਰਿਭਾਸ਼ਿਤ।


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-ਸਟਾਈਲ ਟੂਲਜ਼ ਨਾਲ ਏਜੰਟ ਬਣਾਉਣ


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## ਪ੍ਰੋਡਕਸ਼ਨ ਵਿੱਚ MCP

ਪ੍ਰੋਡਕਸ਼ਨ ਵਾਤਾਵਰਨ ਵਿੱਚ, MCP ਸ਼ਕਤੀਸ਼ালী ਪੈਟਰਨਸ ਨੂੰ ਯੋਗ ਬਣਾਉਂਦਾ ਹੈ:

- **ਡਾਇਨੇਮਿਕ ਟੂਲ ਖੋਜ**: ਏਜੰਟ MCP ਸਰਵਰਾਂ ਨਾਲ ਜੁੜਦੇ ਹਨ ਅਤੇ ਰਨਟਾਇਮ 'ਤੇ ਟੂਲ ਖੋਜਦੇ ਹਨ
- **ਅਲੱਗ-ਅਲੱਗ ਧਾਂਚਾ**: ਟੂਲ ਪ੍ਰਦਾਤਾ ਏਜੰਟ ਤੋਂ ਪਰੇਤਨ ਤੌਰ 'ਤੇ ਅੱਪਡੇਟ ਕਰ ਸਕਦੇ ਹਨ
- **ਸੰਗਠਨ-ਕਰਾਸ ਸਾਂਝਾ ਕਰਨ ਦੀ ਯੋਗਤਾ**: ਟੀਮਾਂ MCP ਸਰਵਰਾਂ ਰਾਹੀਂ ਯੋਗਤਾਵਾਂ ਦਾ ਪ੍ਰਗਟਾਵਾ ਕਰ ਸਕਦੀਆਂ ਹਨ ਜਿਹੜੀ ਕਿਸੇ ਵੀ ਏਜੰਟ ਵੱਲੋਂ ਵਰਤੀ ਜਾ ਸਕਦੀ ਹੈ
- **Microsoft Agent Framework ਸਹਾਇਤਾ**: MAF ਨੇ `mcp` ਇੰਟੀਗ੍ਰੇਸ਼ਨ ਰਾਹੀਂ ਬਿਲਟ-ਇਨ MCP ਕਲਾਇੰਟ ਸਹਾਇਤਾ ਮੁਹੱਈਆ ਕਰਵਾਈ ਹੈ

MAF ਨਾਲ ਇਕ ਅਸਲੀ MCP ਸਰਵਰ ਵਰਤਣ ਲਈ, ਤੁਸੀਂ `hosted_mcp_tool()` ਜਾਂ MCP ਕਲਾਇੰਟ ਇੰਟੀਗ੍ਰੇਸ਼ਨ ਰਾਹੀਂ ਜੁੜੋਗੇ।

**ਹੋਰ ਜਾਣੋ:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## ਸਾਰ

ਇਸ ਪਾਠ ਵਿੱਚ, ਤੁਸੀਂ ਸਿਖਿਆ:
- **MCP** ਏਜੰਟਾਂ ਅਤੇ ਟੂਲ ਪ੍ਰਦਾਤਾਵਾਂ ਦੇ ਵਿਚਕਾਰ गतਿ ਸਕੰਦ ਟੂਲ ਖੋਜ ਲਈ ਇੱਕ ਖੁੱਲਾ ਮਿਆਰ ਹੈ
- **ਕਲਾਈਂਟ-ਸਰਵਰ ਆਰਕੀਟੈਕਚਰ** ਏਜੰਟਾਂ ਨੂੰ ਰਨਟਾਇਮ ਵਿੱਚ ਯੋਗਤਾਵਾਂ ਦੀ ਖੋਜ ਕਰਨ ਦਿੰਦਾ ਹੈ
- MCP **ਵਿਸਤਾਰਯੋਗ, ਅਲੱਗ-ਥੱਲਗ ਏਜੰਟ ਪ੍ਰਣਾਲੀਆਂ** ਨੂੰ ਯਕੀਨੀ ਬਣਾਉਂਦਾ ਹੈ ਜਿੱਥੇ ਕੋਡ ਬਦਲਾਅ ਬਿਨਾਂ ਟੂਲ ਜੋੜੇ ਜਾ ਸਕਦੇ ਹਨ
- ਮਾਇਕਰੋਸਾਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ ਉਤਪਾਦਨ ਲਈ **ਬਿਲਟ-ਇਨ MCP ਸਹਾਇਤਾ** ਪ੍ਰਦਾਨ ਕਰਦਾ ਹੈ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
